# Data Science Tool Mastery Project

**Goal:** demonstrate data cleaning, analysis, visualization, a baseline machine-learning model, SQL and a Power BI plan using a synthetic retail sales dataset.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data' / 'retail_sales_raw.csv', parse_dates=['date'])
df.head()

## 1. Inspect and clean the data
Two categorical values are intentionally missing so that data-cleaning methods can be demonstrated.

In [ ]:
print(df.shape)
print(df.isna().sum())
for col in ['region', 'channel']:
    df[col] = df[col].fillna(df[col].mode().iloc[0])
df['revenue'] = df['units_sold'] * df['unit_price']
df['cost'] = df['units_sold'] * df['unit_cost']
df['profit'] = df['revenue'] - df['cost']
df['profit_margin'] = np.where(df['revenue'] > 0, df['profit'] / df['revenue'], 0)
df['month'] = df['date'].dt.to_period('M').astype(str)
df.head()

## 2. Summary statistics and aggregation

In [ ]:
display(df[['units_sold','revenue','cost','profit','profit_margin']].describe().round(2))
category_summary = df.groupby('category', as_index=False).agg(units_sold=('units_sold','sum'), revenue=('revenue','sum'), profit=('profit','sum')).sort_values('revenue', ascending=False)
display(category_summary)
monthly_summary = df.groupby('month', as_index=False).agg(revenue=('revenue','sum'), profit=('profit','sum'))
display(monthly_summary)

## 3. Visualize the data with Seaborn and Matplotlib

In [ ]:
sns.set_theme()
fig, ax = plt.subplots(figsize=(8,4))
sns.barplot(data=category_summary, x='category', y='revenue', ax=ax)
ax.set_title('Revenue by Product Category'); plt.show()
fig, ax = plt.subplots(figsize=(8,4))
sns.lineplot(data=monthly_summary, x='month', y='revenue', marker='o', ax=ax)
ax.set_title('Monthly Revenue Trend'); plt.show()

## 4. Machine-learning baseline
A Random Forest regressor estimates transaction profit from region, category, product, units, cost, price and channel. Because the dataset is synthetic and very small, this is a learning demonstration only.

In [ ]:
features = ['region','category','product','units_sold','unit_cost','unit_price','channel']
X, y = df[features], df['profit']
cat = ['region','category','product','channel']; num = ['units_sold','unit_cost','unit_price']
pre = ColumnTransformer([('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat),('num', SimpleImputer(strategy='median'), num)])
model = Pipeline([('preprocessor', pre), ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
model.fit(X_train, y_train); pred = model.predict(X_test)
print('MAE:', round(mean_absolute_error(y_test, pred), 2))
print('RMSE:', round(mean_squared_error(y_test, pred)**0.5, 2))
print('R2:', round(r2_score(y_test, pred), 3))

## 5. Conclusions and next steps
Use the category and monthly summaries to identify stronger sales areas and periods. SQL examples are in `sql/analysis_queries.sql`. The Power BI dashboard build steps and DAX measures are in `powerbi/dashboard_guide.md`. For a stronger predictive model, collect more real observations, use a time-aware validation strategy where appropriate, and compare against a simple baseline.